# Biomedical Image Segmentation with U-Net

Lien de Dataset :
    https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset

In [1]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from glob import glob
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.models import Model

In [2]:
import os, sys, subprocess
from glob import glob

# Local (Mac/PC) : si le dossier "Cancer" existe déjà, on l'utilise.
# Colab : sinon, on télécharge le dataset BUSI depuis Kaggle.
if os.path.isdir("Cancer/malignant") and os.path.isdir("Cancer/benign"):
    folder_paths = ["Cancer/malignant", "Cancer/benign"]
else:
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    root = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
    print("Dataset téléchargé dans :", root)
    folder_paths = []
    for name in ("malignant", "benign"):
        found = sorted(glob(os.path.join(root, "**", name), recursive=True))
        if not found:
            raise FileNotFoundError(f"Dossier '{name}' introuvable dans {root}")
        folder_paths.append(found[0])

for p in folder_paths:
    print(p, "->", len(glob(p + "/*")), "fichiers (images + masques)")

Using Colab cache for faster access to the 'breast-ultrasound-images-dataset' dataset.
Dataset téléchargé dans : /kaggle/input/breast-ultrasound-images-dataset
/kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT/malignant -> 421 fichiers (images + masques)
/kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT/benign -> 891 fichiers (images + masques)


In [3]:
# folder_paths est défini dans la cellule précédente (dossiers malignant et benign)

size = 128 # Input size: 128x128

images = [] # Empty list to store the original images in
masks = [] # Empty list to store the masks in

found_mask = False # This flag helps us handle multiple masks for the same image
# Loop through both folders
for folder_path in folder_paths:
    # Loop through all files in the current folder (sorted for consistency)
    for file_path in sorted(glob(folder_path + "/*")):
        # Load and resize the image
        img = cv2.imread(file_path)
        img = cv2.resize(img, (size, size)) # Resize image to 128×128
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) # Convert RGB to grayscale
        img = img / 255.0 # Normalize to [0,1]

        if "mask" in file_path: # Checks if the filename contains "mask"
            if found_mask:
                # Combine with the previous mask
                masks[-1] += img
                # Ensure binary output (0 or 1)
                masks[-1] = np.where(masks[-1] > 0.5, 1.0, 0.0)
            else:
                masks.append(img) # Adds the first mask to the list
                found_mask = True
        else:
            images.append(img) # Adds original image to the list
            found_mask = False

# Vérification : si aucune image n'est trouvée, les chemins ci-dessus sont faux
assert len(images) > 0, f"Aucune image trouvée dans {folder_paths} : vérifiez le chemin du dataset."

# Convert lists to NumPy arrays
X = np.array(images) # Create an array of all original images
y = np.array(masks) # Create an array of all masked imaged (Ground truth)

X = np.expand_dims(X, -1)
y = np.expand_dims(y, -1)

print(f"X shape: {X.shape} | y shape: {y.shape}")
# Split images into training and validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1)

X shape: (647, 128, 128, 1) | y shape: (647, 128, 128, 1)


### Ajouter du code pour visualiser les cartes des caractéristiques (feature maps) de chaque couche dans le modèle U-Net

In [4]:
# Dictionnaire qui contiendra, pour chaque couche à visualiser, son tenseur de sortie.
# Il est rempli pendant la construction du modèle (cellule suivante).
feature_layers = {}

def show_feature_maps(sample, max_maps=16, ncols=8):
    """Affiche les cartes de caractéristiques (feature maps) de toutes les couches
    enregistrées dans `feature_layers`, pour UNE image `sample` de forme (size, size, 1)."""
    # Un seul modèle à plusieurs sorties -> une seule passe avant pour toutes les couches.
    # Il partage les poids du U-Net (donc les poids entraînés si on l'appelle après model.fit).
    activation_model = Model(inputs=input_layer, outputs=list(feature_layers.values()))
    activations = activation_model.predict(sample[np.newaxis, ...], verbose=0)

    for name, act in zip(feature_layers.keys(), activations):
        fmap = act[0]                                  # (H, W, C)
        H, W, C = fmap.shape
        n = min(max_maps, C)
        # On garde les n canaux les plus "actifs" (variance spatiale la plus élevée)
        idx = np.argsort(fmap.reshape(-1, C).var(axis=0))[::-1][:n]

        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(2 * ncols, 2.2 * nrows))
        axes = np.atleast_2d(axes)
        for k, ax in enumerate(axes.flat):
            ax.axis("off")
            if k < n:
                ax.imshow(fmap[..., idx[k]], cmap="viridis")
                ax.set_title(f"canal {idx[k]}", fontsize=8)
        fig.suptitle(f"{name}  |  {H}x{W}x{C}  (top {n} canaux sur {C})", fontsize=11)
        plt.tight_layout()
        plt.show()

In [5]:
# Build the U-Net
input_layer = Input(shape=(size, size, 1))
conv1 = Conv2D(64, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(input_layer)
conv1 = Conv2D(64, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv1)
pool1 = MaxPooling2D((2, 2))(conv1)

# Visualisation : cartes des caractéristiques de la couche 1 de l'encodeur
feature_layers["Encodeur - couche 1"] = conv1

# Second encoder block
conv2 = Conv2D(128, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(pool1)
conv2 = Conv2D(128, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv2)
pool2 = MaxPooling2D((2, 2))(conv2)

# Visualisation : cartes des caractéristiques de la couche 2 de l'encodeur
feature_layers["Encodeur - couche 2"] = conv2

# Third encoder block
conv3 = Conv2D(256, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(pool2)
conv3 = Conv2D(256, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv3)
pool3 = MaxPooling2D((2, 2))(conv3)

# Visualisation : cartes des caractéristiques de la couche 3 de l'encodeur
feature_layers["Encodeur - couche 3"] = conv3

# Fourth encoder block
conv4 = Conv2D(512, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(pool3)
conv4 = Conv2D(512, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv4)
pool4 = MaxPooling2D((2, 2))(conv4)

# Visualisation : cartes des caractéristiques de la couche 4 de l'encodeur
feature_layers["Encodeur - couche 4"] = conv4

# --- Bottleneck ---
bottleneck = Conv2D(1024, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(pool4)
bottleneck = Conv2D(1024, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(bottleneck)

# Visualisation : cartes des caractéristiques de la couche espace latente
feature_layers["Espace latent (bottleneck)"] = bottleneck

# First decoder block
upconv1 = Conv2DTranspose(512, (2, 2), strides=2, padding="same",kernel_initializer="he_normal")(bottleneck)
concat1 = concatenate([upconv1, conv4])
conv5 = Conv2D(512, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(concat1)
conv5 = Conv2D(512, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv5)

# Visualisation : cartes des caractéristiques de la couche 1 du decodeur
feature_layers["Décodeur - couche 1"] = conv5

# Second decoder block
upconv2 = Conv2DTranspose(256, (2, 2), strides=2, padding="same",kernel_initializer="he_normal")(conv5)
concat2 = concatenate([upconv2, conv3])
conv6 = Conv2D(256, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(concat2)
conv6 = Conv2D(256, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv6)

# Visualisation : cartes des caractéristiques de la couche 2 du decodeur
feature_layers["Décodeur - couche 2"] = conv6

# Third decoder block
upconv3 = Conv2DTranspose(128, (2, 2), strides=2, padding="same",kernel_initializer="he_normal")(conv6)
concat3 = concatenate([upconv3, conv2])
conv7 = Conv2D(128, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(concat3)
conv7 = Conv2D(128, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv7)

# Visualisation : cartes des caractéristiques de la couche 3 du decodeur
feature_layers["Décodeur - couche 3"] = conv7

# Fourth decoder block
upconv4 = Conv2DTranspose(64, (2, 2), strides=2, padding="same",kernel_initializer="he_normal")(conv7)
concat4 = concatenate([upconv4, conv1])
conv8 = Conv2D(64, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(concat4)
conv8 = Conv2D(64, (3, 3), activation="relu", padding="same",kernel_initializer="he_normal")(conv8)

# Visualisation : cartes des caractéristiques de la couche 4 du decodeur
feature_layers["Décodeur - couche 4"] = conv8

# --- Output layer ---
output_layer = Conv2D(1, (1, 1), activation="sigmoid", padding="same")(conv8)

# --- Model creation ---
model = Model(inputs=input_layer, outputs=output_layer)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 16, 16,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 8, 8, 512) │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 8, 8,      │  4,719,616 │ max_pooling2d_3[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 8, 8,      │  9,438,208 │ conv2d_8[0][0]    │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 16, 16,    │  2,097,664 │ conv2d_9[0][0]    │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ conv2d_transpose

 Total params: 31,030,593 (118.37 MB)

 Trainable params: 31,030,593 (118.37 MB)

 Non-trainable params: 0 (0.00 B)

![unet.png](attachment:unet.png)

In [ ]:
# Compile and train the model
model.compile(loss="binary_crossentropy", optimizer="Adam", metrics=["accuracy"])
model.fit(X_train, y_train, epochs = 40, validation_data = (X_val,y_val),verbose=1)


Epoch 1/40
 6/19 ━━━━━━━━━━━━━━━━━━━━ 12:14 56s/step - accuracy: 0.5272 - loss: 0.7999

### Visualisation des cartes de caractéristiques (après entraînement)

Pour une image de validation, on affiche les 16 canaux les plus actifs de chaque couche.
- **Encodeur** : les premières couches détectent des contours et des textures ; plus on descend, plus la résolution diminue et les cartes deviennent abstraites.
- **Espace latent** : résolution minimale (8x8), représentation la plus compacte.
- **Décodeur** : la résolution remonte et, grâce aux *skip connections*, les cartes retrouvent les détails spatiaux jusqu'à la localisation de la lésion.

In [ ]:
i = 6                      # indice de l'image de validation (à changer pour tester d'autres images)
sample = X_val[i]          # forme (size, size, 1)

# Image d'entrée et masque réel pour référence
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1); plt.imshow(sample, cmap="gray"); plt.title("Entrée"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(y_val[i], cmap="gray"); plt.title("Vérité terrain"); plt.axis("off")
plt.show()

# Cartes de caractéristiques de chaque couche (encodeur, bottleneck, décodeur)
show_feature_maps(sample, max_maps=16)

In [ ]:
# Compute IoU
from sklearn.metrics import jaccard_score
# Compute predicted mask
pred=model.predict(X_val,verbose=1)
pred = (pred > 0.5).astype(int) # binarize
y_true = y_val.astype(int)
# Compute IoU based on flatten predictions and ground truths
iou = jaccard_score(pred.flatten(), y_true.flatten())
print(f" IoU (Jaccard Score): {iou:.4f}")


In [ ]:
# Plot
i=6 # Try other values
plt.subplot(1, 3, 1)
plt.imshow(X_val[i],cmap="gray")
plt.subplot(1, 3, 2)
plt.imshow(y_val[i],cmap="gray")
plt.subplot(1, 3, 3)
pred=model.predict(np.expand_dims(X_val[i], axis=0),verbose=1)[0]
pred = (pred > 0.5) # binarize
plt.imshow(pred,cmap="gray")

In [ ]:
n = 10
plt.figure(figsize=(10, n * 2))

for i in range(n):
    # Image d'entrée
    plt.subplot(n, 3, 3*i + 1)
    plt.imshow(X_val[i], cmap='gray')
    plt.title("Entrée")
    plt.axis('off')

    # Masque réel
    plt.subplot(n, 3, 3*i + 2)
    plt.imshow(y_val[i], cmap='gray')
    plt.title("Vérité terrain")
    plt.axis('off')

    # Prédiction
    pred = model.predict(np.expand_dims(X_val[i], axis=0), verbose=0)[0]
    pred = (pred > 0.5)

    plt.subplot(n, 3, 3*i + 3)
    plt.imshow(pred, cmap='gray')
    plt.title("Prédiction")
    plt.axis('off')

plt.tight_layout()
plt.show()